[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch01-introduction/02-hanh-vi-nam-trong-du-lieu.ipynb)

# Hành vi nằm trong dữ liệu: khoảng trống kiểm chứng, nhãn hỏng và suy giảm âm thầm

Chương 1 mô tả bước chuyển từ phần mềm 1.0 sang phần mềm 2.0 (Software 2.0): mã nguồn của phần mềm 2.0 là dữ
liệu huấn luyện cùng với nhãn, trình biên dịch là vòng lặp huấn luyện, logic thì ẩn và được học. Hệ quả mà
chương nhấn mạnh là cách hệ thống thất bại: phần mềm 1.0 thất bại ồn ào, còn phần mềm 2.0 thất bại im lặng,
chỉ số tụt dần mà không có dòng log nào báo.

Sổ tay này làm từng ý đó thành một phép tính hay một thí nghiệm nhỏ trên MNIST.

**Bạn sẽ làm:**
1. tính khoảng trống kiểm chứng của ImageNet đúng tới từng chữ số, và xem nó mở ra từ ảnh nhỏ cỡ nào;
2. đo độ chính xác của một mô hình cùng khoảng tin cậy (confidence interval) của nó;
3. giữ nguyên mã huấn luyện, chỉ đổi quy trình gán nhãn, và thấy chương trình đổi theo;
4. thử hai cách sửa: lấy mô hình làm trung tâm (model-centric) và lấy dữ liệu làm trung tâm;
5. cho ánh sáng đổi dần qua 12 tháng: độ chính xác tụt trong khi dịch vụ không báo một lỗi nào; khớp phương
   trình suy giảm, đặt ngưỡng cảnh báo, và thử một đòn bẩy làm đường suy giảm thoải hơn;
6. thấy cùng một triệu chứng có thể đến từ hai trục khác nhau của D·A·M;
7. đổi phần trăm ra số lần bỏ sót, rồi tính theo dõi bao lâu một lần thì phát hiện trễ bao nhiêu.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd
import torch.nn.functional as F
import torchvision
from scipy.stats import wasserstein_distance

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định, như các sổ tay khác

tap = torchvision.datasets.MNIST(DATA, train=True, download=True)
assert len(tap) == 60_000
anh = tap.data.float().div(255).reshape(-1, 784)
nhan = tap.targets.clone()

# Mọi phần đều lấy từ tập huấn luyện của MNIST, để "kiểm tra" và "lưu lượng thật" có cùng phân phối với dữ
# liệu huấn luyện. Dịch chuyển phân phối duy nhất trong sổ tay là cái phần 5 tự đưa vào.
X, y = anh[:10_000], nhan[:10_000]                      # huấn luyện
X_kt, y_kt = anh[50_000:55_000], nhan[50_000:55_000]    # kiểm tra trước khi triển khai
X_song, y_song = anh[55_000:], nhan[55_000:]            # lưu lượng sau khi triển khai
print(f"huấn luyện {len(X):,} ảnh · kiểm tra {len(X_kt):,} ảnh · lưu lượng thật lấy từ {len(X_song):,} ảnh")

## 1. Khoảng trống kiểm chứng

Logic của phần mềm 1.0 thường liệt kê được, chia thành các đường đi và các ca biên. Một hệ thống đã học thì
hoạt động trên một không gian đầu vào hữu hạn về lý thuyết nhưng không thể liệt kê trong thực tế. Chương lấy
cuộc thi ImageNet năm 2012 làm ví dụ: mỗi đầu vào là một ảnh RGB 224 × 224, mỗi kênh 8 bit, tức 256 mức.

**Dự đoán:** số cấu hình điểm ảnh khả dĩ có bao nhiêu chữ số? Vài chục, vài nghìn, hay hàng trăm nghìn?

In [ ]:
CANH = sach(224, nguon="ảnh 224×224 RGB")
KENH = 3          # đỏ, lục, lam
MUC = 2 ** 8      # mỗi kênh 8 bit
gia_tri = CANH * CANH * KENH
assert gia_tri == 150_528
chu_so = math.floor(gia_tri * math.log10(MUC)) + 1
theo_sach("số chữ số của 256 luỹ thừa 150,528", chu_so, sach=362_508, nguon="số có 362.508 chữ số")

# Kiểm lại bằng số nguyên chính xác của Python, không qua số thực
khong_gian = MUC ** gia_tri
assert 10 ** (chu_so - 1) <= khong_gian < 10 ** chu_so

VALIDATION = sach(50_000, nguon="tập validation ILSVRC chỉ 50.000 ảnh")
mu_phu = math.log10(VALIDATION) - gia_tri * math.log10(MUC)
print(f"{VALIDATION:,} ảnh phủ khoảng 10^{mu_phu:,.0f} của không gian đầu vào")
print(f"khoảng trống kiểm chứng = tổng không gian − phần được phủ ≈ tổng không gian "
      f"(lệch nhau ở chữ số thứ {chu_so - len(str(VALIDATION)) + 1:,} trở đi)")

Phương trình 1 của chương viết đúng điều đó: khoảng trống kiểm chứng bằng tổng không gian đầu vào trừ phần
bộ kiểm thử phủ được, và gần như bằng chính tổng không gian đầu vào.

Khoảng trống này không cần ảnh lớn mới mở ra. Đồ thị dưới đây đếm số chữ số của không gian đầu vào theo cạnh
ảnh, cho ảnh đen trắng (1 bit mỗi điểm ảnh), ảnh xám 8 bit như MNIST, và ảnh màu 8 bit như ImageNet, rồi đặt
cạnh số chữ số của 50,000.

In [ ]:
cac_canh = np.array([1, 2, 3, 4, 5, 8, 16, 28, 64, 128, 224])
kieu = {"đen trắng, 1 bit": (1, 2), "xám 8 bit (như MNIST)": (1, 256), "màu 8 bit (như ImageNet)": (3, 256)}
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for (ten, (k, muc)), mau in zip(kieu.items(), MAU):
    ax.loglog(cac_canh, cac_canh ** 2 * k * math.log10(muc) + 1, "o-", color=mau, lw=2, label=ten)
ax.axhline(math.log10(VALIDATION) + 1, color="0.4", ls="--", lw=1)
ax.text(40, (math.log10(VALIDATION) + 1) * 0.62, "số chữ số của 50,000", fontsize=8, color="0.3", ha="center")
mnist = 28 * 28 * math.log10(256)
ax.annotate(f"MNIST 28 × 28: {math.floor(mnist) + 1:,} chữ số", (28, mnist + 1), xytext=(-150, 18),
            textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.annotate(f"ImageNet 224 × 224: {chu_so:,} chữ số", (224, chu_so), xytext=(-215, -4), textcoords="offset points",
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xticks(cac_canh, [str(c) for c in cac_canh], fontsize=8)
ax.minorticks_off()
ax.set_ylim(0.8, 3e6)
ax.set_xlabel("cạnh ảnh, điểm ảnh (thang log)")
ax.set_ylabel("số chữ số của số đầu vào khả dĩ (thang log)")
ax.set_title("Khoảng trống kiểm chứng mở ra từ ảnh rất nhỏ")
ax.legend(loc="upper left", bbox_to_anchor=(0, 0.92), fontsize=8)
plt.tight_layout()
plt.show()
print(f"ảnh đen trắng 4 × 4 đã có {2 ** 16:,} cấu hình, nhiều hơn {VALIDATION:,}")

## 2. Thứ mà một tập kiểm tra chứng minh được

Khoảng trống này không làm kiểm thử mất ý nghĩa, mà đổi điều kiểm thử khẳng định được. Đánh giá trước khi
triển khai cho **bằng chứng thống kê** trên những đầu vào đã lấy mẫu: độ chính xác đo được, kèm một khoảng
tin cậy. Khẳng định đó nói về cả một tổng thể đầu vào, và chỉ đúng chừng nào đầu vào thật còn đến từ cùng tổng
thể ấy.

Mô hình của sổ tay là mạng 784-128-64-10 quen thuộc, huấn luyện 5 epoch trên 10,000 ảnh. Hàm `huan_luyen` dưới
đây đóng vai trình biên dịch của phần mềm 2.0: nó biến dữ liệu và nhãn thành một chương trình.

In [ ]:
def mlp(rong=(128, 64)):
    lop, vao = [], 784
    for r in rong:
        lop += [torch.nn.Linear(vao, r), torch.nn.ReLU()]
        vao = r
    return torch.nn.Sequential(*lop, torch.nn.Linear(vao, 10))


def huan_luyen(X, y, rong=(128, 64), so_epoch=5, tang_sang=0.0):
    """Vòng lặp huấn luyện: dữ liệu và nhãn vào, chương trình ra. Cùng hạt giống thì cùng kết quả."""
    dat_hat_giong()
    mo_hinh = mlp(rong)
    toi_uu = torch.optim.Adam(mo_hinh.parameters(), lr=1e-3)
    for _ in range(so_epoch):
        thu_tu = torch.randperm(len(X))
        for i in range(0, len(X), 32):
            chon = thu_tu[i:i + 32]
            xb = X[chon]
            if tang_sang:  # tăng cường dữ liệu: làm sáng nền mỗi ảnh một lượng ngẫu nhiên (dùng ở phần 5)
                b = tang_sang * torch.rand(len(chon), 1)
                xb = b + (1 - b) * xb
            mat_mat = F.cross_entropy(mo_hinh(xb), y[chon])
            toi_uu.zero_grad()
            mat_mat.backward()
            toi_uu.step()
    return mo_hinh


def du_doan(mo_hinh, X):
    with torch.no_grad():
        return mo_hinh(X).argmax(1)


def do_chinh_xac(mo_hinh, X, y):
    return (du_doan(mo_hinh, X) == y).float().mean().item()


def so_tham_so(mo_hinh):
    return sum(p.numel() for p in mo_hinh.parameters())


t0 = time.perf_counter()
mo_hinh = huan_luyen(X, y)
do_tren_may("thời gian huấn luyện", time.perf_counter() - t0, "s")

p = do_chinh_xac(mo_hinh, X_kt, y_kt)
nua = 1.96 * math.sqrt(p * (1 - p) / len(X_kt))  # khoảng tin cậy 95 % cho một tỉ lệ
do_tren_may(f"độ chính xác trên {len(X_kt):,} ảnh kiểm tra", 100 * p, "%")
do_tren_may("nửa khoảng tin cậy 95 %", 100 * nua, "điểm phần trăm")
do_tren_may(f"cùng độ chính xác, nếu đo trên {VALIDATION:,} ảnh như ILSVRC, nửa khoảng tin cậy",
            100 * 1.96 * math.sqrt(p * (1 - p) / VALIDATION), "điểm phần trăm")

Khoảng tin cậy hẹp lại theo căn bậc hai của số ảnh: gấp mười lần số ảnh chỉ thu khoảng đó lại hơn ba lần. Đó
là cái giá của độ tin cậy thống kê, thứ thay thế cho kỳ vọng về một chứng minh vét cạn. Và khoảng đó chỉ nói về
những ảnh giống tập kiểm tra. Phần 5 sẽ cho thấy điều gì xảy ra khi tổng thể đầu vào đổi.

## 3. Dữ liệu là mã nguồn

Giả định của sổ tay: 5,000 ảnh cuối của tập huấn luyện là một lô mới, gán nhãn bằng một quy trình mới, và
công cụ mới có một lỗi: mọi chữ số 7 đều bị ghi thành nhãn 1. Mã huấn luyện không đổi một dòng nào. Chỉ có
nhãn đổi.

**Dự đoán:** mô hình mới khác mô hình cũ ở bao nhiêu phần trăm dự đoán, và khác ở những ảnh nào?

In [ ]:
lo_moi = torch.arange(len(y)) >= 5_000          # lô thứ hai, quy trình gán nhãn mới
y_moi = y.clone()
y_moi[lo_moi & (y == 7)] = 1                     # giả định: công cụ mới ghi mọi chữ số 7 thành nhãn 1
print(f"{int((y_moi != y).sum())} nhãn trên {len(y):,} ảnh đã đổi; ảnh và mã huấn luyện giữ nguyên")

mo_hinh_moi = huan_luyen(X, y_moi)               # đúng hàm cũ, đúng hạt giống cũ
d_cu, d_moi = du_doan(mo_hinh, X_kt), du_doan(mo_hinh_moi, X_kt)
khac = d_cu != d_moi
do_tren_may("dự đoán khác nhau giữa hai chương trình", 100 * khac.float().mean().item(), "%")
do_tren_may("trong số dự đoán khác nhau, phần là ảnh chữ số 7", 100 * (y_kt[khac] == 7).float().mean().item(), "%")
print(f"để so: ảnh chữ số 7 chiếm {100 * (y_kt == 7).float().mean().item():.1f} % tập kiểm tra")

la_7 = y_kt == 7
fig, ax = plt.subplots(figsize=(7.5, 3.6))
rong_cot = 0.38
for k, (ten, d, mau) in enumerate((("nhãn cũ", d_cu, MAU[0]), ("nhãn mới, cùng mã", d_moi, MAU[1]))):
    ti_le = torch.bincount(d[la_7], minlength=10).float() / la_7.sum()
    ax.bar(np.arange(10) + (k - 0.5) * rong_cot, 100 * ti_le.numpy(), rong_cot, color=mau, label=ten)
ax.set_xticks(range(10))
ax.set_xlabel("chữ số mà mô hình đoán")
ax.set_ylabel("% ảnh chữ số 7 thật")
ax.set_title("Ảnh chữ số 7 được đoán thành gì (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

So phần chữ số 7 trong các dự đoán khác nhau với phần chữ số 7 trong cả tập kiểm tra, hai dòng in ở trên: sự khác
biệt dồn về đúng chỗ dữ liệu đã đổi, và biểu đồ cho thấy nó dồn về đâu. Cùng một vòng lặp huấn luyện, cùng hạt
giống, cùng ảnh, nhưng hai chương trình khác nhau. Đó là nguyên lý dữ liệu là mã nguồn: dữ liệu huấn luyện
không chỉ là đầu vào của một chương trình cố định, mà góp phần quyết định logic mà mô hình thực thi. Không một
công cụ so sánh mã nào thấy được thay đổi này, vì thứ thay đổi không phải một dòng mã mà kỹ sư viết.

## 4. Sửa ở mô hình hay sửa ở dữ liệu?

Chương gọi tên hai cách làm. Lấy mô hình làm trung tâm thì giữ dữ liệu cố định và sửa mô hình. Lấy dữ liệu làm
trung tâm thì giữ mã tương đối cố định và cải thiện dữ liệu một cách có hệ thống.

**Dự đoán:** một mạng rộng gấp đôi, huấn luyện lâu gấp đôi trên cùng dữ liệu, có cứu được chữ số 7 không?

In [ ]:
to = huan_luyen(X, y_moi, rong=(256, 256), so_epoch=10)
print(f"mạng 784-256-256-10: {so_tham_so(to):,} tham số, gấp {so_tham_so(to) / so_tham_so(mo_hinh_moi):.1f} lần "
      f"mạng 784-128-64-10; huấn luyện lại 10 epoch trên đúng nhãn hỏng")

Cách thứ hai bắt đầu bằng gỡ lỗi dữ liệu thay vì gỡ lỗi mã: thay vì lần theo đường thực thi, hãy xem phân phối
của dữ liệu. Đếm nhãn theo từng lô là phép kiểm rẻ nhất:

In [ ]:
dem = pd.DataFrame({"lô 1": torch.bincount(y_moi[~lo_moi], minlength=10).numpy(),
                    "lô 2": torch.bincount(y_moi[lo_moi], minlength=10).numpy()})
dem.index.name = "nhãn"
print(dem.T.to_string())

Lô 2 không có một nhãn 7 nào và có gần gấp đôi nhãn 1: lỗi nằm ở quy trình gán nhãn của lô 2. Bước sửa theo
kiểu lấy dữ liệu làm trung tâm là rà lại mọi ảnh mang nhãn 1 trong lô 2. Giả định của sổ tay: người rà sửa
đúng hết. Rồi huấn luyện lại bằng đúng mạng nhỏ và đúng mã cũ.

In [ ]:
can_ra = lo_moi & (y_moi == 1)
y_sua = y_moi.clone()
y_sua[can_ra] = y[can_ra]                     # người rà gán lại nhãn đúng
assert torch.equal(y_sua, y)                  # nhãn trở lại đúng như trước khi có lỗi
sua = huan_luyen(X, y_sua)


def flop_huan_luyen(mo_hinh, so_anh, so_epoch):
    """Xấp xỉ huấn luyện dày đặc: 6 FLOP cho mỗi tham số, mỗi ảnh, mỗi epoch (như sổ tay 01)."""
    return 6 * so_tham_so(mo_hinh) * so_anh * so_epoch


bang = []
for ten, m, epoch, ra in (("nhãn hỏng, mạng 784-128-64-10", mo_hinh_moi, 5, 0),
                          ("lấy mô hình làm trung tâm: 784-256-256-10, 10 epoch", to, 10, 0),
                          ("lấy dữ liệu làm trung tâm: rà nhãn 1 của lô 2", sua, 5, int(can_ra.sum()))):
    d = du_doan(m, X_kt)
    bang.append({"cách làm": ten, "GFLOP huấn luyện": flop_huan_luyen(m, len(X), epoch) / 1e9,
                 "nhãn phải rà": ra,
                 "độ chính xác chung, đo (%)": 100 * (d == y_kt).float().mean().item(),
                 "chữ số 7 đoán đúng, đo (%)": 100 * (d[la_7] == 7).float().mean().item()})
bang = pd.DataFrame(bang)
for _, r in bang.iterrows():
    do_tren_may(f"{r['cách làm']}: độ chính xác chung", r["độ chính xác chung, đo (%)"], "%")
    do_tren_may(f"{r['cách làm']}: chữ số 7 đoán đúng", r["chữ số 7 đoán đúng, đo (%)"], "%")
print(bang.round(1).to_string(index=False))

Đọc bảng theo hai cột cuối, không theo cột độ chính xác chung. Trong dữ liệu hỏng, khoảng một nửa số chữ số 7
mang nhãn 7, nửa kia mang nhãn 1. Từ chính dữ liệu đó, không mô hình nào, lớn tới đâu, học được rằng chữ số 7 là
7 một cách nhất quán: một mạng lớn hơn chỉ học trung thành hơn cái logic mà dữ liệu dạy nó. Vì vậy cột chữ số 7
của nó vẫn hỏng, dù độ chính xác chung nhích lên hay tụt xuống.

Rà lại mọi nhãn 1 của lô 2 thì nhãn trở lại đúng như cũ, và mạng nhỏ học lại được chữ số 7, với một phần năm
số phép tính huấn luyện của cách thứ nhất. Cái giá của cách này nằm ở cột khác: hơn một nghìn nhãn phải có người
xem lại.

Đó là lý do chương nói cải thiện riêng mô hình không sửa được một quy trình dữ liệu bị hỏng, và vì sao gỡ lỗi
một hệ thống học máy là gỡ lỗi cả dữ liệu, không chỉ các đoạn mã Python.

## 5. Suy giảm âm thầm

Giờ hãy triển khai mô hình sạch ở phần 2 và cho thế giới đổi dần. Ví dụ của chương là hệ thống hỗ trợ người lái:
mùa đổi mang theo điều kiện chiếu sáng ít xuất hiện trong dữ liệu huấn luyện. Phiên bản MNIST của nó, giả định
của sổ tay: mỗi tháng, nền ảnh sáng thêm 1 % thang xám. Mỗi tháng có 2,000 yêu cầu, lấy ngẫu nhiên từ phần lưu
lượng thật.

Dịch vụ dưới đây làm đúng những gì một dịch vụ phục vụ dự đoán làm: nhận ảnh, trả nhãn và độ tin cậy. Mỗi tháng
sổ tay ghi lại bốn thứ: số lỗi mà dịch vụ ném ra, độ tin cậy trung bình, độ chính xác (khi nhãn thật về tới), và
một độ phân kỳ thống kê giữa đầu vào tháng đó với đầu vào huấn luyện.

Độ phân kỳ ở đây là khoảng cách Wasserstein, một trong những lựa chọn chương nêu, giữa hai phân phối độ sáng
trung bình của từng ảnh. Nó không cần nhãn.

**Dự đoán:** độ chính xác tụt tới đâu trước khi dịch vụ ném ra lỗi đầu tiên?

In [ ]:
SANG_MOI_THANG = 0.01        # giả định: nền sáng thêm 1 % thang xám mỗi tháng
YEU_CAU = 2_000              # yêu cầu mỗi tháng
sang_huan_luyen = X.mean(1).numpy()


def phuc_vu(mo_hinh, anh):
    """Dịch vụ: ảnh vào, (nhãn, độ tin cậy) ra."""
    with torch.no_grad():
        xac_suat = F.softmax(mo_hinh(anh), dim=1)
    tin_cay, nhan_doan = xac_suat.max(1)
    return nhan_doan, tin_cay


def phan_ky(anh):
    """Độ phân kỳ D(P_t ‖ P_0): khoảng cách Wasserstein giữa độ sáng trung bình từng ảnh, không cần nhãn."""
    return wasserstein_distance(anh.mean(1).numpy(), sang_huan_luyen)


def luu_luong(thang, rng):
    chon = torch.from_numpy(rng.choice(len(X_song), YEU_CAU, replace=False))
    b = SANG_MOI_THANG * thang
    return b + (1 - b) * X_song[chon], y_song[chon]


ben = huan_luyen(X, y, tang_sang=0.15)   # đòn bẩy thứ hai, dùng ở dưới: tăng cường dữ liệu khi huấn luyện

rng = np.random.default_rng(SEED)
nhat_ky = []
for thang in range(13):
    anh_thang, nhan_that = luu_luong(thang, rng)
    so_loi = 0
    try:
        nhan_doan, tin_cay = phuc_vu(mo_hinh, anh_thang)
    except Exception:
        so_loi += 1
    assert nhan_doan.min() >= 0 and nhan_doan.max() <= 9   # mọi câu trả lời đều là một nhãn hợp lệ
    nhat_ky.append({"tháng": thang, "lỗi ném ra": so_loi, "độ tin cậy TB": tin_cay.mean().item(),
                    "độ chính xác": (nhan_doan == nhan_that).float().mean().item(),
                    "độ chính xác, mô hình tăng cường": (phuc_vu(ben, anh_thang)[0] == nhan_that).float().mean().item(),
                    "phân kỳ D": phan_ky(anh_thang)})
nhat_ky = pd.DataFrame(nhat_ky)
print(nhat_ky.round(3).to_string(index=False))
assert nhat_ky["lỗi ném ra"].sum() == 0

Cột "lỗi ném ra" bằng 0 suốt mười hai tháng, và mọi câu trả lời đều là một nhãn hợp lệ: hai điều đó được `assert`,
không phụ thuộc máy. Một bảng điều khiển chỉ theo dõi lỗi và độ trễ sẽ xanh cả năm. Đó là suy giảm âm thầm: thuật
toán vẫn chạy, máy vẫn phục vụ dự đoán, nhưng hành vi đã học thì ngày càng kém chính xác.

Độ tin cậy trung bình là thứ duy nhất dịch vụ tự thấy mà không cần nhãn, nhưng nó không phải độ chính xác: nó đo
mức mô hình tự tin, không đo mức mô hình đúng. Hãy so hai cột đó trong bảng. Độ chính xác chỉ biết được khi nhãn
thật về tới.

### Phương trình suy giảm, khớp cục bộ

Chương đưa ra một quan hệ chẩn đoán:

$$\text{Accuracy}(t) \approx \text{Accuracy}_0 - \lambda\cdot\mathcal{D}(P_t \,\|\, P_0)$$

và nói rõ đó là xấp xỉ chẩn đoán cục bộ, không phải định luật dự đoán phổ quát: nó có thể vỡ khi dịch chuyển
lớn. Hãy khớp $\text{Accuracy}_0$ và $\lambda$ trên sáu tháng đầu, rồi xem đường thẳng đó đi tiếp ra sao.

In [ ]:
THANG_KHOP = 6
dau = nhat_ky[nhat_ky["tháng"] < THANG_KHOP]
doc, chan = np.polyfit(dau["phân kỳ D"], 100 * dau["độ chính xác"], 1)
lam = -doc
do_tren_may("Accuracy_0 khớp được", chan, "%")
do_tren_may("λ khớp được, mô hình gốc", lam, "điểm phần trăm mỗi đơn vị D")
for cot, ten in (("độ chính xác", "mô hình gốc"), ("độ chính xác, mô hình tăng cường", "mô hình tăng cường")):
    doc_nam = np.polyfit(nhat_ky["phân kỳ D"], 100 * nhat_ky[cot], 1)[0]
    do_tren_may(f"λ khớp trên cả năm, {ten}", -doc_nam, "điểm phần trăm mỗi đơn vị D")

# Ngưỡng cảnh báo: phải cao hơn mức dao động tự nhiên của D khi chưa có dịch chuyển nào
rng_nen = np.random.default_rng(SEED + 1)
nen = [phan_ky(luu_luong(0, rng_nen)[0]) for _ in range(30)]
TAU = 2 * max(nen)   # giả định: gấp đôi giá trị lớn nhất của 30 tháng không dịch chuyển
thang_bao = int(nhat_ky.loc[nhat_ky["phân kỳ D"] > TAU, "tháng"].min())
do_tren_may("mức dao động lớn nhất của D khi không dịch chuyển", max(nen), "")
do_tren_may("tháng đầu tiên D vượt ngưỡng τ", thang_bao, "")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.plot(nhat_ky["tháng"], 100 * nhat_ky["độ chính xác"], "o-", color=MAU[0], lw=2, label="độ chính xác (cần nhãn)")
ax1.plot(nhat_ky["tháng"], 100 * nhat_ky["độ tin cậy TB"], "s--", color=MAU[3], lw=2, label="độ tin cậy trung bình")
ax1.plot(nhat_ky["tháng"], 100 * nhat_ky["độ chính xác, mô hình tăng cường"], "^:", color=MAU[2], lw=2,
         label="độ chính xác, mô hình tăng cường dữ liệu")
ax1.axvline(thang_bao, color=MAU[1], lw=1.2, label="tháng đầu tiên D vượt ngưỡng τ")
ax1.set_ylim(50, 100)
ax1.set_xlabel("tháng sau khi triển khai")
ax1.set_ylabel("%")
ax1.set_title("Không một lỗi nào được ném ra (đo trên máy này)")
ax1.legend(loc="lower left", fontsize=8)

x_duong = np.linspace(0, nhat_ky["phân kỳ D"].max(), 50)
ax2.plot(nhat_ky["phân kỳ D"], 100 * nhat_ky["độ chính xác"], "o", color=MAU[0], ms=6, label="đo trên máy này")
ax2.plot(x_duong, chan + doc * x_duong, "--", color=MAU[1], lw=2,
         label=f"Accuracy₀ − λ·D, khớp trên {THANG_KHOP} tháng đầu")
ax2.axvspan(0, dau["phân kỳ D"].max(), color=MAU[1], alpha=0.08)
ax2.axvline(TAU, color="0.4", ls=":", lw=1.2, label="ngưỡng cảnh báo τ")
ax2.set_ylim(50, 100)
ax2.set_xlabel("độ phân kỳ D(P_t ‖ P_0)")
ax2.set_ylabel("độ chính xác (%)")
ax2.set_title("Xấp xỉ tuyến tính đúng ở gần, vỡ ở xa")
ax2.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Đầu tiên, đường thẳng: so các điểm đo với nó trong vùng tô màu, nơi nó được khớp, rồi ngoài vùng đó. Một xấp xỉ
bậc nhất khớp ở gần không có lý do gì để đúng ở xa, và chương dặn đúng điều đó: quan hệ này có thể vỡ khi dịch
chuyển lớn, và độ phân kỳ một mình không quyết định dấu hay độ lớn của mức thay đổi độ chính xác.

Chương nói phương trình gợi ý ba đòn bẩy kỹ thuật. Đọc theo từng số hạng của nó:

* **$\text{Accuracy}_0$**: một mô hình tốt hơn lúc triển khai nâng cả đường lên, nhưng không đổi độ dốc.
* **$\lambda$**: mô hình thứ hai được huấn luyện với tăng cường dữ liệu (data augmentation), mỗi ảnh huấn luyện
  được làm sáng nền một lượng ngẫu nhiên. Phân phối huấn luyện rộng hơn, phủ cả những ảnh sáng hơn, là cách làm
  cho cùng một $\mathcal{D}$ lấy đi ít điểm hơn. Hãy so hai giá trị $\lambda$ khớp trên cả năm ở trên, và đường
  chấm màu lục trên biểu đồ bên trái.
* **$\mathcal{D}$**: theo dõi độ trôi. Độ phân kỳ không cần nhãn, nên đọc được ngay trong tháng, không phải chờ
  nhãn thật về. Nó có thể vượt ngưỡng $\tau$ khi độ chính xác chưa nhúc nhích. Đó là lý do chương đề xuất hai bước
  tách rời: cảnh báo khi $\mathcal{D}$ vượt $\tau$, và huấn luyện lại khi bằng chứng về kết quả xác nhận là đã suy
  giảm.

Ngưỡng $\tau$ không thể đặt tuỳ ý: nó phải cao hơn mức dao động tự nhiên của $\mathcal{D}$ giữa các tháng không
có dịch chuyển nào, nếu không hệ thống sẽ báo động giả mỗi tháng.

Một điều nữa: độ phân kỳ ở đây nhìn vào độ sáng, đúng chiều mà sổ tay làm dịch chuyển. Một dịch chuyển theo chiều
khác, chẳng hạn chữ số lệch sang một bên, giữ nguyên độ sáng và sẽ không hiện lên ở thước đo này. Chọn thước đo
độ trôi là một quyết định thiết kế, không phải một việc làm một lần cho mọi trường hợp. Thử thêm 2 cho bạn xem
điều đó.

## 6. Cùng một triệu chứng, hai trục khác nhau

Phân loại D·A·M của chương nói cùng một triệu chứng mà người dùng thấy có thể bắt nguồn từ trục dữ liệu, trục
thuật toán hay trục máy, và mỗi giả thuyết đòi một loại bằng chứng khác. Hãy dựng hai hệ thống có cùng triệu
chứng: độ chính xác trên lưu lượng thật thấp hơn hẳn lúc kiểm tra.

* Hệ thống D: mô hình tốt ở phần 2, ở tháng mà độ chính xác của nó gần nhất với hệ thống A.
* Hệ thống A: một mô hình quá nhỏ, 784-6-10, trên lưu lượng không dịch chuyển.

In [ ]:
nho = huan_luyen(X, y, rong=(6,), so_epoch=3)
anh_0, nhan_0 = luu_luong(0, np.random.default_rng(SEED + 2))
dung_nho = do_chinh_xac(nho, anh_0, nhan_0)
k = int((nhat_ky["độ chính xác"] - dung_nho).abs().idxmin())
anh_k, nhan_k = luu_luong(k, np.random.default_rng(SEED + 3))

chan_doan = pd.DataFrame([
    {"hệ thống": f"D: mô hình 784-128-64-10, tháng {k}",
     "độ chính xác, lưu lượng thật (%)": 100 * do_chinh_xac(mo_hinh, anh_k, nhan_k),
     "độ chính xác, chính tập huấn luyện (%)": 100 * do_chinh_xac(mo_hinh, X, y),
     "phân kỳ D": phan_ky(anh_k)},
    {"hệ thống": "A: mô hình 784-6-10, tháng 0",
     "độ chính xác, lưu lượng thật (%)": 100 * dung_nho,
     "độ chính xác, chính tập huấn luyện (%)": 100 * do_chinh_xac(nho, X, y),
     "phân kỳ D": phan_ky(anh_0)},
])
for _, r in chan_doan.iterrows():
    do_tren_may(f"{r['hệ thống']}: độ chính xác trên lưu lượng thật", r["độ chính xác, lưu lượng thật (%)"], "%")
print(chan_doan.round(3).to_string(index=False))

Triệu chứng giống nhau, nhưng bằng chứng chỉ về hai hướng khác nhau. Hệ thống D vẫn đúng gần hết trên chính tập
huấn luyện, còn độ phân kỳ thì cao: giả thuyết dữ liệu, cần xem độ bao phủ và dịch chuyển phân phối. Hệ thống A
sai nhiều ngay trên dữ liệu nó đã học, còn độ phân kỳ nằm ở mức nền: giả thuyết thuật toán, cần xem dung lượng mô
hình. Thêm dữ liệu sẽ không cứu được A, còn mạng lớn hơn sẽ không cứu được D.

Trục thứ ba, máy, cần một loại bằng chứng nữa: độ trễ, thông lượng, bộ nhớ và mức sử dụng. Sổ tay 01 làm phần đó.

## 7. Từ phần trăm sang số lần

### Một xe chạy một giờ

Chương đổi mức suy giảm của hệ thống hỗ trợ người lái ra đơn vị khác: một mô hình nhận thức chạy 10 Hz xử lý
36,000 khung hình trong một giờ, và độ chính xác phát hiện người đi bộ có thể tụt từ 95 % xuống 85 %.

In [ ]:
HZ = sach(10, nguon="mô hình chạy 10 Hz")
theo_sach("khung hình trong một giờ", HZ * 3600, sach=36_000, nguon="xử lý 36.000 khung trong một giờ")
TRUOC = sach(95, nguon="giảm từ 95 % xuống 85 % qua vài tháng")
SAU = sach(85, nguon="giảm từ 95 % xuống 85 % qua vài tháng")
BO_SOT = sach(0.1, nguon="tỉ lệ bỏ sót 0,1 %")
for ten, phat_hien in (("lúc triển khai", TRUOC), ("vài tháng sau", SAU)):
    print(f"{ten}: phát hiện {phat_hien} %, bỏ sót {1000 * (100 - phat_hien) / 100:.0f} trên mỗi 1,000 ca liên quan")
print(f"bỏ sót tăng {(100 - SAU) / (100 - TRUOC):.0f} lần; tỉ lệ bỏ sót {BO_SOT} % là {1000 * BO_SOT / 100:.0f} trên 1,000 ca")

Ở đây "độ chính xác phát hiện" được đọc là tỉ lệ người đi bộ được phát hiện trong những ca có người đi bộ, cách
đọc của sổ tay. Mười điểm phần trăm nghe nhỏ, nhưng số lần bỏ sót tăng gấp ba. Chương dặn thêm hai điều: tỉ lệ
bỏ sót 0.1 % chỉ áp cho những ca dương tính liên quan, nên số lần bỏ sót mỗi giờ còn tuỳ tần suất của những ca đó,
vào việc lọc theo thời gian và hợp nhất cảm biến. Sổ tay không đoán tần suất ấy, nên chỉ đếm trên mỗi 1,000 ca.

### Một hệ gợi ý, sáu tháng

Trong một ví dụ minh hoạ của chương, một hệ gợi ý triển khai ở mức 85 % đã tụt xuống 80.2 % trong 6 tháng, khi
thói quen mua sắm thay đổi, mà không đổi một dòng mã nào.

In [ ]:
GOI_Y_0 = sach(85, nguon="hệ gợi ý 85 % → 80,2 % trong 6 tháng")
GOI_Y_6 = sach(80.2, nguon="hệ gợi ý 85 % → 80,2 % trong 6 tháng")
SO_THANG = sach(6, nguon="hệ gợi ý 85 % → 80,2 % trong 6 tháng")
theo_sach("số điểm mất", GOI_Y_0 - GOI_Y_6, sach=4.8, nguon="mất 4,8 điểm phần trăm")
diem_moi_ngay = (GOI_Y_0 - GOI_Y_6) / (SO_THANG * 30)   # giả định: tụt đều, tháng 30 ngày
print(f"nếu tụt đều: {30 * diem_moi_ngay:.1f} điểm mỗi tháng, {diem_moi_ngay:.3f} điểm mỗi ngày")

### Theo dõi bao lâu một lần?

Giả sử mỗi $I$ ngày, hệ thống chạy một lần kiểm tra độ chính xác. Suy giảm bắt đầu vào một thời điểm bất kỳ giữa
hai lần kiểm tra. Nếu thời điểm đó rơi đều trong khoảng, độ trễ phát hiện trung bình là $I/2$ và tối đa là $I$.
Kiểm tra độ chính xác còn phải chờ nhãn thật về tới, một độ trễ nhãn $L$ ngày (giả định của sổ tay).

**Dự đoán:** kiểm tra hằng tháng thay vì hằng tuần thì hệ gợi ý mất thêm bao nhiêu điểm trước khi ai đó biết?

In [ ]:
rng = np.random.default_rng(SEED)
for I in (7, 30):
    bat_dau = rng.uniform(0, I, 100_000)            # suy giảm bắt đầu ở đâu đó giữa hai lần kiểm tra
    tre = I - bat_dau                                # chờ tới lần kiểm tra kế tiếp
    print(f"kiểm tra mỗi {I:>2} ngày: trễ trung bình {tre.mean():.2f} ngày (lý thuyết {I / 2}), tối đa {tre.max():.2f}")

cac_I = np.array([1, 3, 7, 14, 30, 60, 90])
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for L, mau, kieu_net in ((0, MAU[0], "o-"), (7, MAU[1], "s--"), (14, MAU[2], "^:")):
    ax.plot(cac_I, diem_moi_ngay * (cac_I / 2 + L), kieu_net, color=mau, lw=2, label=f"độ trễ nhãn {L} ngày")
ax.set_xscale("log")
ax.set_xticks(cac_I, [str(i) for i in cac_I])
ax.minorticks_off()
ax.set_xlabel("chu kỳ kiểm tra I (ngày, thang log)")
ax.set_ylabel("điểm đã mất lúc phát hiện, trung bình")
ax.set_title("Hệ gợi ý tụt 4.8 điểm trong 6 tháng, nếu tụt đều")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()
for I in (7, 30):
    print(f"mỗi {I} ngày: {365 / I:.0f} lần kiểm tra một năm, mất trung bình {diem_moi_ngay * (I / 2 + 7):.2f} điểm "
          f"với độ trễ nhãn 7 ngày")

Kiểm tra dày hơn rút ngắn độ trễ phát hiện, đổi lại nhiều lần kiểm tra hơn, nhưng nó không sửa được độ chính
xác: theo dõi chỉ quan sát kết quả, không sửa mô hình. Và khi nhãn về chậm, độ trễ nhãn đặt một mức sàn mà kiểm
tra dày tới đâu cũng không xuống dưới được. Đó là chỗ độ phân kỳ ở phần 5, thứ không cần nhãn, có giá trị: nó báo
sớm để đi tìm bằng chứng, trước khi nhãn kịp về.

## Thử thêm

1. Ở phần 3, đổi lỗi gán nhãn thành `y_moi[lo_moi & (y == 7) & (torch.rand(len(y)) < 0.3)] = 1`, tức chỉ 30 % chữ
   số 7 của lô 2 bị ghi sai. Mô hình còn đoán sai chữ số 7 nhiều không? Phép đếm nhãn theo lô ở phần 4 còn bắt
   được lỗi rõ như trước không?
2. Ở phần 5, trong `luu_luong`, thay dòng `return b + (1 - b) * X_song[chon], y_song[chon]` bằng
   `return torch.roll(X_song[chon].reshape(-1, 28, 28), thang // 3, dims=2).reshape(-1, 784), y_song[chon]`:
   mỗi ba tháng chữ số lệch sang phải thêm một điểm ảnh, còn độ sáng giữ nguyên. Độ chính xác làm gì, và $D$ có
   vượt $\tau$ không? Mô hình tăng cường còn đỡ được không?
3. Ở phần 6, đổi `rong=(6,)` thành `(3,)` hoặc `(16,)`. Tháng nào của hệ thống D giờ có cùng triệu chứng, và bằng
   chứng còn chỉ đúng trục không?

## Tóm lại

* Ảnh RGB 224 × 224 có $256^{150{,}528}$ cấu hình, một số có 362,508 chữ số; 50,000 ảnh validation phủ một phần
  nhỏ tới mức không viết ra được. Kiểm thử cho bằng chứng thống kê về một tổng thể, không cho chứng minh.
* Cùng một mã huấn luyện, hai quy trình gán nhãn, cho ra hai chương trình khác nhau: dữ liệu là mã nguồn.
* Khi lỗi nằm ở dữ liệu, mô hình lớn hơn học lại lỗi đó trung thành hơn; rà lại dữ liệu mới sửa được, và gỡ lỗi
  bắt đầu bằng phân phối của dữ liệu, không phải đường thực thi của mã.
* Dịch chuyển phân phối làm độ chính xác tụt mà không ném ra lỗi nào; phương trình suy giảm chỉ đúng cục bộ, độ
  phân kỳ không cần nhãn nên báo được sớm, và tăng cường dữ liệu làm đường suy giảm thoải hơn.
* Mười điểm phần trăm có thể là gấp ba số lần bỏ sót; theo dõi thưa hơn và nhãn về chậm hơn là thêm điểm mất trước
  khi có ai biết.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Nhập môn hệ thống học máy](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch01-introduction/), dựng từ chương
[*Introduction*](https://mlsysbook.ai/vol1/introduction/introduction.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_01_ml_intro.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.